
# Agentic RAG Chatbot

## Projekt Áttekintés

Ez a notebook egy **Agentic RAG (Retrieval-Augmented Generation)** chatbot prototípust
valósít meg.
Részletesebb leírás a README.md fájlban

A notebook lépésről-lépésre végigmegy a projekten, bemutatva a teljes pipeline egyes komponenseit, felépítését.

# Környezet beállításai

In [11]:
pip install -r requirements.txt


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from src.config import print_config
print_config()

22:02:19 Konfiguráció betöltve:
22:02:19   LLM Provider: ollama
22:02:19   LLM Model:    mistral:7b-instruct-v0.3-q4_K_M
22:02:19   Embedding:    sentence-transformers/all-MiniLM-L6-v2
22:02:19   Chunk:        1000 chars, 200 overlap
22:02:19   Top-K:        4
22:02:19   PDF források: 5 dokumentum
22:02:19   HF Token:     van


# Dokumentum letöltés, feldolgozás

In [3]:
from src.document_processing import download_pdfs, load_all_documents, chunk_documents, print_chunk_stats

print("PDF-ek letöltése...")
pdf_paths = download_pdfs()

print(f"\nSzöveg kinyerés...")
all_docs = load_all_documents(pdf_paths)
print(f"{len(all_docs)} oldal\n")

chunks = chunk_documents(all_docs)
print_chunk_stats(chunks)

22:02:23 Attention Is All You Need (Vaswani et al., 2017) már letöltve
22:02:23 Retrieval-Augmented Generation for Knowledge-Intensive NLP (Lewis et al., 2020) már letöltve
22:02:23 Chain-of-Thought Prompting (Wei et al., 2022) már letöltve
22:02:23 Corrective Retrieval Augmented Generation (Yan et al., 2024) már letöltve
22:02:23 Agentic RAG: A Survey (Singh et al., 2025) már letöltve
22:02:23 Attention Is All You Need (Vaswani et al., 2017): 15 oldal
22:02:23 Retrieval-Augmented Generation for Knowledge-Intensive NLP (Lewis et al., 2020): 19 oldal


PDF-ek letöltése...

Szöveg kinyerés...


22:02:23 Chain-of-Thought Prompting (Wei et al., 2022): 43 oldal
22:02:23 Corrective Retrieval Augmented Generation (Yan et al., 2024): 16 oldal
22:02:23 Agentic RAG: A Survey (Singh et al., 2025): 39 oldal
22:02:23 Chunk-olás: 542 chunk
22:02:23 Átlag: 880, Min: 157, Max: 1000
22:02:23 Forrás szerint:
22:02:23   Chain-of-Thought Prompting (Wei et al., 2022): 179
22:02:23   Agentic RAG: A Survey (Singh et al., 2025): 139
22:02:23   Retrieval-Augmented Generation for Knowledge-Intensive NLP (Lewis et al., 2020): 92
22:02:23   Corrective Retrieval Augmented Generation (Yan et al., 2024): 80
22:02:23   Attention Is All You Need (Vaswani et al., 2017): 52


132 oldal



# Embeddelés és Vector DB

- **Embedder**: `all-MiniLM-L6-v2` (384 dim) – gyors, lokális, jó ár/érték
- **Vector DB**: ChromaDB – egyszerű, persistent, LangChain-kompatibilis

In [4]:
from src.vector_store import create_embedding_model, create_vectorstore, get_retriever, test_retriever

embedding_model = create_embedding_model()
vectorstore = create_vectorstore(chunks, embedding_model)
retriever = get_retriever(vectorstore)

test_retriever(retriever, [
    "How does the attention mechanism work in transformers?",
    "What is retrieval-augmented generation?",
    "How does chain-of-thought prompting improve reasoning?",
    "What are the key components of agentic RAG systems?",
])


22:02:27 Embedding modell: sentence-transformers/all-MiniLM-L6-v2...
22:02:27 Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
22:02:30 Betöltve (dimenzió: 384)
22:02:30 Vector store építés (542 chunk)...
22:02:30 Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
22:02:36 542 dokumentum indexelve
22:02:36 Retrieval teszt:
22:02:36   Q: How does the attention mechanism work in transformers?
22:02:36     [1] Attention Is All You Need (Vaswani et al., 2017) (p.5): Applications of Attention in our Model The Transformer uses multi-head attention in three different ...
22:02:36     [2] Attention Is All You Need (Vaswani et al., 2017) (p.3): 3.2 Attention An attention function can be described as mapping a query and a set of key-value pairs...
22:02:36     [3] Attention Is All You Need (Vaswani et al., 2017) (p.3): Figure 1: The Transformer - model architecture. The Transformer follows this overall archi

# LLM teszt

In [5]:
from src.llm import test_llm
test_llm()

22:02:39 LLM teszt (ollama)...
22:02:48 Válasz: Hello, working!


# Agent Node-ok – Egyenkénti Teszt

Minden node-ot önállóan tesztelünk, mielőtt a gráfba kötjük.

In [6]:
from src.nodes import (
    make_initial_state, route_question, make_retrieve_node,
    grade_documents, rewrite_query, generate_response, check_hallucination,
)

# Router teszt
print("═══ Router Node ═══\n")
for q, expected in [
    ("How does self-attention work?", "retrieve"),
    ("Hello!", "direct"),
    ("What is corrective RAG?", "retrieve"),
    ("What is 2+2?", "direct"),
]:
    s = route_question(make_initial_state(q))
    ok = "OK" if s["route_decision"] == expected else "NOT OK"
    print(f"{ok} '{q}' → {s['route_decision']} (expected: {expected})\n")

# Retriever + Grader teszt
print("═══ Retriever + Grader ═══\n")
_retrieve = make_retrieve_node(retriever)
state = _retrieve(make_initial_state("What is the transformer architecture?"))
state = grade_documents(state)
print(f"\n Releváns dokumentumok: {len(state['documents'])}")

# Query Rewriter teszt
print("═══ Query Rewriter ═══\n")
for q in ["How do AI agents work?", "Is RAG useful?", "What makes models smarter?"]:
    rewrite_query(make_initial_state(q))
    print()

# 5.4 Generator + Hallucination Check teszt
print("═══ Generator + Hallucination Checker ═══\n")
state = generate_response(state)
state = check_hallucination(state)

═══ Router Node ═══



22:02:49 Router: retrieve
22:02:50 Router: direct


OK 'How does self-attention work?' → retrieve (expected: retrieve)

OK 'Hello!' → direct (expected: direct)



22:02:50 Router: retrieve


OK 'What is corrective RAG?' → retrieve (expected: retrieve)



...

22:02:59 Rewriter: 'How do AI agents work?' → 'What is the functioning mechanism of transformer-based AI models, particularly focusing on their attention mechanisms and their role in retrieval-augmented generation (RAG)?'


22:03:01 Rewriter: 'Is RAG useful?' → 'What is the effectiveness of Retrieval-Augmented Generation (RAG) systems in AI research?'


22:03:02 Rewriter: 'What makes models smarter?' → 'How does the implementation of advanced techniques such as transformers and attention mechanisms in AI models contribute to their increased intelligence?'



═══ Generator + Hallucination Checker ═══



22:03:17 Generator (#1):  The Transformer architecture (Vaswani et al., 2017) consists of stacked self-attention and point-wise, fully connected layers for both the encoder an...
22:03:23 Hallucination: OK (grounded)


# LangGraph Gráf & Demo

A teljes Agentic RAG pipeline.

In [7]:
from src.graph import build_graph, run_agent, measure_latency

app = build_graph(retriever)

try:
    print(app.get_graph().draw_mermaid())
except Exception:
    print("(Vizualizáció nem elérhető")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	retrieve(retrieve)
	grade_docs(grade_docs)
	rewrite_query(rewrite_query)
	generate(generate)
	direct_generate(direct_generate)
	check_hallucination(check_hallucination)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	check_hallucination -. &nbsp;end&nbsp; .-> __end__;
	check_hallucination -. &nbsp;regenerate&nbsp; .-> generate;
	generate --> check_hallucination;
	grade_docs -.-> generate;
	grade_docs -. &nbsp;rewrite&nbsp; .-> rewrite_query;
	retrieve --> grade_docs;
	rewrite_query --> retrieve;
	router -. &nbsp;direct&nbsp; .-> direct_generate;
	router -.-> retrieve;
	direct_generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Demo 1: Egyszerű ténykérdés

In [8]:
run_agent(app, "What is the transformer architecture and how does self-attention work?")

22:03:23 ======================================================================
22:03:23 What is the transformer architecture and how does self-attention work?
22:03:23 ======================================================================
22:03:25 Router: retrieve
22:03:25 Retriever: 4 doc (próba #1)
22:03:30 Grader: 3/4 releváns
22:03:44 Generator (#1):  The Transformer architecture, as described in Vaswani et al. (2017), consists of stacked self-attention and point-wise, fully connected layers for bo...
22:03:51 Hallucination: OK (grounded)
22:03:51 ──────────────────────────────────────────────────────────────────────
22:03:51 VÁLASZ:
 The Transformer architecture, as described in Vaswani et al. (2017), consists of stacked self-attention and point-wise, fully connected layers for both the encoder and decoder. The encoder is composed of a series of identical layers, each containing two sub-layers: a multi-head self-attention mechanism and a simple, position-wise fully connected feed

{'question': 'What is the transformer architecture and how does self-attention work?',
 'generation': ' The Transformer architecture, as described in Vaswani et al. (2017), consists of stacked self-attention and point-wise, fully connected layers for both the encoder and decoder. The encoder is composed of a series of identical layers, each containing two sub-layers: a multi-head self-attention mechanism and a simple, position-wise fully connected feed-forward network. Self-attention in the Transformer is an attention mechanism that relates different positions of a single sequence to compute a representation of the sequence. It has been used successfully for various tasks such as reading comprehension, abstractive summarization, textual entailment, and learning task-independent sentence representations (Vaswani et al., 2017).',
 'documents': [Document(metadata={'total_pages': 15, 'page': 5, 'source': 'attention_is_all_you_need', 'title': 'Attention Is All You Need (Vaswani et al., 2017

### Demo2: Nem releváns kérdés

In [9]:
run_agent(app, "Hello, what is your name?")

22:03:51 ======================================================================
22:03:51 Hello, what is your name?
22:03:51 ======================================================================
22:03:52 Router: direct
22:03:55 Direct:  Hello! I don't have a physical form or personal identity, but I'm here to assist you. I can certainly help with questions related to AI research pape...
22:03:55 ──────────────────────────────────────────────────────────────────────
22:03:55 VÁLASZ:
 Hello! I don't have a physical form or personal identity, but I'm here to assist you. I can certainly help with questions related to AI research papers. What specific question or topic are you interested in today?
22:03:55 ──────────────────────────────────────────────────────────────────────
22:03:55 Route=direct, Retrieval×0, Generate×0, Docs=0


{'question': 'Hello, what is your name?',
 'generation': " Hello! I don't have a physical form or personal identity, but I'm here to assist you. I can certainly help with questions related to AI research papers. What specific question or topic are you interested in today?",
 'documents': [],
 'retrieval_count': 0,
 'generation_count': 0,
 'route_decision': 'direct',
 'hallucination_check': ''}

## Demo 3: Összetett kérdés (cross-paper)

In [10]:
run_agent(app, "How does retrieval-augmented generation improve upon the limitations of standard transformer models?")

22:03:55 ======================================================================
22:03:55 How does retrieval-augmented generation improve upon the limitations of standard transformer models?
22:03:55 ======================================================================
22:03:57 Router: retrieve
22:03:57 Retriever: 4 doc (próba #1)
22:04:03 Grader: 2/4 releváns
22:04:18 Generator (#1):  Retrieval-Augmented Generation (RAG) improves upon standard transformer models by supplementing them with a retrieval component that provides relevan...
22:04:25 Hallucination: OK (grounded)
22:04:25 ──────────────────────────────────────────────────────────────────────
22:04:25 VÁLASZ:
 Retrieval-Augmented Generation (RAG) improves upon standard transformer models by supplementing them with a retrieval component that provides relevant knowledge from external documents. This addresses the limitation of transformer models relying solely on their own parameters, as RAG can provide more factual and specific

{'question': 'How does retrieval-augmented generation improve upon the limitations of standard transformer models?',
 'generation': " Retrieval-Augmented Generation (RAG) improves upon standard transformer models by supplementing them with a retrieval component that provides relevant knowledge from external documents. This addresses the limitation of transformer models relying solely on their own parameters, as RAG can provide more factual and specific responses (Lewis et al., 2020). However, it's important to note that if the retrieved documents are irrelevant, the system could potentially exacerbate factual errors. Advanced versions of RAG have been developed to address this issue by selectively using retrieval when necessary (Asai et al., 2024).",
 'documents': [Document(metadata={'total_pages': 19, 'source': 'rag_original', 'title': 'Retrieval-Augmented Generation for Knowledge-Intensive NLP (Lewis et al., 2020)', 'page': 9}, page_content='could represent promising future work.\n6\

# Teljesítmény mérés

In [11]:
print("Latency mérés:\n")
lat = measure_latency(app, "What is the attention mechanism?", n_runs=2)
print(f"\n Átlag: {lat['avg']:.2f}s, Min: {lat['min']:.2f}s, Max: {lat['max']:.2f}s")

Latency mérés:



22:04:27 Router: retrieve
22:04:27 Retriever: 4 doc (próba #1)
22:04:36 Grader: 3/4 releváns
22:04:55 Generator (#1):  The attention mechanism, as described in the paper "Attention Is All You Need" (Vaswani et al., 2017), is a function that maps a query and a set of k...
22:05:06 Hallucination: OK (grounded)
22:05:06 Run 1: 40.86s
22:05:08 Router: retrieve
22:05:08 Retriever: 4 doc (próba #1)
22:05:16 Grader: 3/4 releváns
22:05:38 Generator (#1):  The attention mechanism, as described in the paper "Attention Is All You Need" (Vaswani et al., 2017), is a function that maps a query and a set of k...
22:05:48 Hallucination: OK (grounded)
22:05:48 Run 2: 42.07s



 Átlag: 41.47s, Min: 40.86s, Max: 42.07s


# Bottleneck-ek és teljesítmény

A legnagyobb probléma egyértelműen a lassúság. Egy kérdés megválaszolása kb. 55 másodpercet vesz igénybe, ami a 6 szekvenciális LLM hívásból adódik. Ez egy 7b-os modellt 16GB-os Macbookon, CPU-n futtatva persze várható, viszont éles környezetben nem lenne elfogadható.

Erre a legkézenfekvőbb megoldás, ha a grading és routing taskokhoz kisebb modellt használunk, mivel ezek bináris döntések, nem igényelnek erős reasoning-et. A generáláshoz viszont érdemes megtartani a nagyobb modellt, mert ott számít a koherencia. Egy másik irány, hogy a grader-t teljesen lecseréljük egy cross-encoder reranker-re (pl. `cross-encoder/ms-marco-MiniLM-L-6-v2`), ami LLM hívás nélkül, pusztán embedding szinten dönti el a relevancia-kérdést, ez nagyságrendekkel gyorsabb lenne, viszont veszítene a pontosságból.

A chunk mérettel is lehetne kísérletezni. A jelenlegi 1000 karakteres chunk elég jól működik, de van, hogy az overlap nem elég, és határra esik a "tudás", vagy éppen túl nagy az 1000-es chunk. Erre a semantic chunking lenne a megoldás, ami nem fix karakter-határon, hanem szemantikai töréspontoknál vág, viszont jóval erőforrásigényesebb.

Az embedding modell ehhez a felhasználáshoz jó kompromisszum, de éles rendszerben érdemes lenne egy nagyobbra váltani, ami a benchmarkokon jobban teljesít, így a retrieval minőségét közvetlenül javítaná.

Végül van egy tudatos trade-off a grader és a hallucination checker pontosságában: mindkettő csak a dokumentumok első `grader_doc_preview_chars` (jelenleg 500) karakterét kapja, nem a teljes szöveget. Ennek az az oka, hogy a teljes chunk elküldése minden grading hívásban jelentősen növelné a latency-t (4 dokumentum × teljes szöveg = sok token). A 500 karakter a legtöbb esetben elég a relevancia megítéléséhez, mert a chunk eleje általában tartalmazza a fő gondolatot. De van, ahol a lényeg a chunk közepén vagy végén van, és ilyenkor a grader tévesen szűrhet. Ez a sebesség-pontosság kompromisszum config szinten állítható (`config.json` → `agent.grader_doc_preview_chars`).


# Teljesítmény mérése

Két szinten lehetne mérni, a retrieval és a válaszgenerálás minőségét külön-külön.

A retrieval-hez kézzel össze kellene rakni egy kis ground truth datasetet: 20-30 kérdés, mindegyikhez hozzárendelve melyik paper melyik szekciója tartalmazza a választ. Ebből ki lehet számolni hit rate-et (a top-K-ban benne van-e a helyes dokumentum) és MRR-t (milyen pozícióban van). Ez lenne az első lépés, mert ha a retrieval rossz, a többi hiába jó.

A válasz minőségéhez lehetne a RAGAS framework-öt használnám, ami négy metrikát ad: faithfulness (a válasz megalapozott-e a kontextusban), answer relevancy (releváns-e a kérdésre), context precision és context recall. Ez automatizáltan futtatható, nem kell kézzel annotálni minden választ.

Érdemes lenne azt is feljegyezni, hogy milyen gyakran van szükséga a query rewrite-ra és a regeneration-re, mert ha túl gyakran, az arra utal, hogy a retrieval vagy a prompt nem elég jó.

# Továbbfejlesztési irányok

Ami szerintem a legnagyobb hatással lenne rövid távon: hybrid search. Jelenleg csak vector similarity search van használva, de a BM25 lexikális keresés bizonyos esetekben jobb eredményt ad. 
A kettő kombinációja szinte mindig javít a retrieval minőségen, és nem igényel sok extra munkát.

Kiegészíthető még tudásgráffal is, ami a komplexebb és mélyebb kapcsolatokat is fel tudja tárni, bár ez egy nagyobb fejlesztés lenne.

Segítene még, ha lenne egy conversation memory. A jelenlegi rendszer minden kérdést izoláltan kezel, szóval ha az előző kérdésre adott válasz alapján szeretnék következő kérdést feltenni, azt nem tudná kezelni. Ehhez a LangGraph state-et ki kellene bővíteni egy conversation history mezővel.

Hosszabb távon multi-agent architektúra lenne az igazán érdekes. Jelenleg egyetlen agent kezeli az összes retrieval-t, de elképzelhető, hogy specializált agentek dolgozzanak párhuzamosan, mondjuk egy a paper metaadatokkal (szerző, évszám, citation count), egy a tartalmi kérdésekkel, egy pedig web search fallback-ként.



#  Összefoglalás

Egy Corrective RAG + Adaptive RAG hibrid architektúrát valósítottam meg LangGraph-fal, 5 arXiv paper-en. A rendszer 6 agent node-ból áll, amelyek az agentic design pattern-eket implementálják: routing (autonóm döntéshozatal), reflection (dokumentum grading), planning (query rewrite) és evaluator-optimizer (hallucination check).

A prototípus működik, de a latency és a retrieval minőség terén van hova fejlődni. A kódbázis úgy van felépítve, hogy az LLM, az embedding modell és a dokumentumforrások config-szinten cserélhetők, a node-ok pedig egymástól függetlenül tesztelhetők és módosíthatók.